# Correlazioni dinamiche del trimero a catena — dal circuito alla validazione

Mirror di `correlazioni_trimero_anello_esplorazione.ipynb`: costruzione del circuito
Hadamard test, verifica passo per passo che riproduce i correlatori attesi, convergenza
Trotter, validazione a shot finiti.

Punto di lavoro: $J=1, b=b_c=3.0, D=0.15$ (VQE-DM).

## 0. Setup

In [1]:
import numpy as np
from qiskit_aer import AerSimulator
from qiskit import transpile

from trimer_chain_exact import trimer_hamiltonian_dm
from circuito_correlazioni_trimero_catena import (
    ground_state, build_correlator_circuit, correlator_from_circuit, ANCILLA,
)
from validate_circuito_correlazioni_catena import classical_exact

J, b, D = 1.0, 3.0, 0.15
T, N = 1.3, 200

## 1. Ground state e Hamiltoniana

In [2]:
H = trimer_hamiltonian_dm(J, b, D).to_matrix()
psi0, E = ground_state(J, b, D)
print(f"E0={E[0]:.6f}  gap(E1-E0)={E[1]-E[0]:.6f}")
print(f"norma psi0 = {np.linalg.norm(psi0):.10f}")

E0=-7.369247  gap(E1-E0)=0.734742
norma psi0 = 1.0000000000


## 2. Il circuito con l'ancilla

In [3]:
qc_re = build_correlator_circuit(1, "y", 2, "y", t=1.0, N=2, J=J, b=b, D=D,
                                  part="re", psi0=psi0)
print(qc_re.draw(output="text"))
print(f"\nqubit totali: {qc_re.num_qubits} (3 registro + 1 ancilla, indice {ANCILLA})")

     »
q_0: »
     »
q_1: »
     »
q_2: »
     »
q_3: »
     »
«     ┌─────────────────────────────────────────────────────────────────────────────────────┐»
«q_0: ┤0                                                                                    ├»
«     │                                                                                     │»
«q_1: ┤1 State Preparation(0,0.0070182,0.0070182,0.28937,0.0070182,-0.57873,0.28937,0.7053) ├»
«     │                                                                                     │»
«q_2: ┤2                                                                                    ├»
«     └────────────────────────────────────────┬───┬────────────────────────────────────────┘»
«q_3: ─────────────────────────────────────────┤ H ├─────────────────────────────────────────»
«                                              └───┘                                         »
«                                        ┌─────────┐┌─────────┐        »
«q_0: ───

## 3. Validazione: statevector vs classico esatto

Quattro casi rappresentativi: un'autocorrelazione ricca ($C_{11}^{yy}$), la sua controparte
sul sito equivalente ($C_{33}^{yy}$, deve coincidere per $P_{13}$), un cross-site
($C_{12}^{yy}$), e un caso a zero esatto a $t=0$ ($C_{11}^{xz}$).

In [4]:
cases = [
    (1, "y", 1, "y", 1.3),
    (3, "y", 3, "y", 1.3),
    (1, "y", 2, "y", 2.7),
    (1, "x", 1, "z", 0.0),
]
print(f"{'correlatore':<12}{'t':>5}   {'classico esatto':>20}   {'circuito (N=200)':>20}   {'|residuo|':>10}")
for (i, al, j, be, t) in cases:
    c_ref = classical_exact(i, al, j, be, t, J, b, D, psi0, H)
    c_circ = correlator_from_circuit(i, al, j, be, t, N, J, b, D, psi0)
    resid = abs(c_ref - c_circ)
    lab = f"C_{i}{j}^{al}{be}"
    print(f"{lab:<12}{t:>5.1f}   {c_ref.real:+.4f}{c_ref.imag:+.4f}i        "
          f"{c_circ.real:+.4f}{c_circ.imag:+.4f}i        {resid:.2e}")

correlatore     t        classico esatto       circuito (N=200)    |residuo|


C_11^yy       1.3   +0.3243-0.4919i        +0.3244-0.4922i        3.33e-04


C_33^yy       1.3   +0.3243-0.4919i        +0.3249-0.4918i        6.30e-04


C_12^yy       2.7   +0.1069+0.3946i        +0.1111+0.3858i        9.72e-03


C_11^xz       0.0   +0.0000+0.0000i        +0.0000-0.0000i        4.58e-16


**Discussione.** Il quarto caso ($C_{11}^{xz}(0)$) conferma via circuito lo zero a $t=0$
predetto dal Cor. "zero-t0-same" (non un vincolo per ogni $t$, solo a $t=0$: qui $i=j$, quindi
è l'argomento $\langle\sigma^y\rangle=0$ per stato reale, non lo stesso argomento usato per
$i\neq j$). Il primo e il secondo caso coincidono entro l'errore di Trotter, come previsto
dalla relazione $P_{13}$.

## 4. Convergenza in $N$ (errore di Trotter)

In [5]:
i, al, j, be, t = 1, "y", 1, "y", 1.3
c_ref = classical_exact(i, al, j, be, t, J, b, D, psi0, H)
prev = None
for Nn in [10, 20, 40, 80, 160, 320]:
    c_circ = correlator_from_circuit(i, al, j, be, t, Nn, J, b, D, psi0)
    err = abs(c_ref - c_circ)
    ratio = f"  (rapporto: {prev/err:.2f})" if prev else ""
    print(f"  N={Nn:4d}: errore = {err:.3e}{ratio}")
    prev = err
print("atteso: errore ~ O(1/N), rapporto ~2 raddoppiando N (Trotter 1o ordine)")

  N=  10: errore = 2.080e-03
  N=  20: errore = 1.345e-03  (rapporto: 1.55)
  N=  40: errore = 1.224e-03  (rapporto: 1.10)


  N=  80: errore = 7.493e-04  (rapporto: 1.63)


  N= 160: errore = 4.089e-04  (rapporto: 1.83)


  N= 320: errore = 2.130e-04  (rapporto: 1.92)
atteso: errore ~ O(1/N), rapporto ~2 raddoppiando N (Trotter 1o ordine)


## 5. Validazione a shot finiti

In [6]:
from circuito_correlazioni_trimero_catena import build_correlator_circuit as bcc
BACKEND = AerSimulator()

def shot_estimate(i, al, j, be, t, Nn, shots, part, seed=None):
    qc = bcc(i, al, j, be, t, Nn, J, b, D, part, psi0=psi0, measure=True)
    qc_t = transpile(qc, BACKEND)
    result = BACKEND.run(qc_t, shots=shots, seed_simulator=seed).result()
    counts = result.get_counts()
    n0, n1 = counts.get('0', 0), counts.get('1', 0)
    return (n0 - n1) / shots

i, al, j, be, t = 1, "y", 1, "y", 1.3
shots = 8192
re = shot_estimate(i, al, j, be, t, N, shots, "re", seed=42)
im = shot_estimate(i, al, j, be, t, N, shots, "im", seed=43)
c_sh = re + 1j * im
c_sv = correlator_from_circuit(i, al, j, be, t, N, J, b, D, psi0)
c_ref = classical_exact(i, al, j, be, t, J, b, D, psi0, H)
print(f"classico esatto : {c_ref.real:+.4f}{c_ref.imag:+.4f}i")
print(f"statevector     : {c_sv.real:+.4f}{c_sv.imag:+.4f}i")
print(f"shots ({shots}) : {c_sh.real:+.4f}{c_sh.imag:+.4f}i")

err_trotter = abs(c_sv - c_ref)
soglia = 1 / np.sqrt(shots)
print(f"\nerrore Trotter (statevector vs esatto): {err_trotter:.2e}")
print(f"soglia statistica caso peggiore 1/sqrt(shots): {soglia:.4f}")
print(f"rapporto statistico/Trotter: {soglia/err_trotter:.1f}x")

classico esatto : +0.3243-0.4919i
statevector     : +0.3244-0.4922i
shots (8192) : +0.3347-0.4807i

errore Trotter (statevector vs esatto): 3.33e-04
soglia statistica caso peggiore 1/sqrt(shots): 0.0110
rapporto statistico/Trotter: 33.2x


**Discussione.** Analisi più estesa (convergenza vs $N_\text{shots}$ su molte
ripetizioni, scan completo delle 81 combinazioni) è nello script standalone
`validate_shot_noise_trimero_catena.py` / `scan81_trimero_catena.py` e nel notebook
successivo — qui solo un caso singolo per verificare che il circuito misurato via shot dia
un risultato sensato.

## 6. Riepilogo

- Il circuito riproduce i correlatori classici entro l'errore di Trotter atteso, sia sui casi
  ricchi sia sui casi predetti a zero (a $t=0$).
- Convergenza Trotter confermata $O(1/N)$.
- Misura a shot finiti coerente con statevector + rumore statistico atteso.

Prossimo passo: `circuito_correlazioni_trimero_catena_tutte.ipynb` (tutte le 81 combinazioni,
preparazione esatta), poi la versione con VQE reale.